In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
file_path = 'merged_Results_Spillover_Random_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Average',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex on rows/cols ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations from index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# Make Sampling categorical with desired order
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows by Sampling then MaxSig ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations aligned to sorted index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns (fixed method order) ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors (NO .map; avoid MultiIndex isna path) ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',             
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 10},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02),
    vmin=vmin, vmax=vmax
)

# === Axis styling ===
# Remove row tick labels (2Median_300_500, etc.)
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

# X-axis labels: match annot font size and rotate
g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=10)
g.ax_heatmap.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=3, width=0.5, labelsize=8  # same as annot size
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])  
vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=10, rotation=270, labelpad=15)   
cbar.ax.tick_params(labelsize=6, width=0.4, length=3)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends (shift further left) ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.65, 0.20, 0.20])
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=8, title_fontsize=9)

legend_ax2 = g.fig.add_axes([0.02, 0.45, 0.20, 0.20]) 
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=8, title_fontsize=9)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
g.fig.savefig("Heatmap_Random_Spillover_Final.svg")
plt.close(g.fig)


In [18]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
file_path = 'merged_Results_Spillover_Uniform_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# ------------------ Pivot for heatmap  ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Average',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex on rows/cols ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations from index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# Make Sampling categorical with desired order
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows by Sampling then MaxSig ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations aligned to sorted index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors (NO .map; avoid MultiIndex isna path) ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',             
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 10},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02),
    vmin=vmin, vmax=vmax
)

# === Axis styling ===
# Remove row tick labels (2Median_300_500, etc.)
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=13)
g.ax_heatmap.yaxis.set_label_position("left")

# X-axis labels: match annot font size and rotate
g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=13)
g.ax_heatmap.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6, labelsize=10  # same as annot size
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])  
vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=12, width=0.4, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.65, 0.20, 0.20])  
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=10, title_fontsize=11)

legend_ax2 = g.fig.add_axes([0.02, 0.45, 0.20, 0.20]) 
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=10, title_fontsize=11)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
g.fig.savefig("Heatmap_Uniform_Spillover_Final.svg")
plt.close(g.fig)

In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
file_path = 'merged_Results_Spillover_Absent_Random_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# ------------------ Pivot for heatmap  ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Average',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex on rows/cols ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# Build annotations from index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# Make Sampling categorical with desired order
annotations['Sampling'] = pd.Categorical(annotations['Sampling'], categories=sampling_order, ordered=True)

# ------------------ Sort rows by Sampling then MaxSig ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# Recompute annotations aligned to sorted index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Sampling'] = pd.Categorical(idx_str.str.split('_').str[0], categories=sampling_order, ordered=True)
annotations['MaxSig']   = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns (fixed method order) ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors (NO .map; avoid MultiIndex isna path) ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series   = pd.Series(annotations['MaxSig'].astype(int).values,   index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig':   [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',             
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 10},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02),
    vmin=vmin, vmax=vmax
)

# === Axis styling ===
# Remove row tick labels (2Median_300_500, etc.)
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=13)
g.ax_heatmap.yaxis.set_label_position("left")

# X-axis labels: match annot font size and rotate
g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=13)
g.ax_heatmap.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6, labelsize=10  
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])  
vmin = np.nanmin(heatmap_data.to_numpy())
vmax = np.nanmax(heatmap_data.to_numpy())
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=12, width=0.4, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.20, 0.20])  # moved left
legend_ax1.axis('off')
legend_ax1.legend(handles=sampling_patches, title='Sampling', loc='center', fontsize=10, title_fontsize=11)

legend_ax2 = g.fig.add_axes([0.02, 0.40, 0.20, 0.20])  # moved left
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=10, title_fontsize=11)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
g.fig.savefig("Heatmap_Random_Spillover_Absent_Final.svg")
plt.close(g.fig)